# dataclasses-replace-args — ex2: 2-D grid sweep over (lr, batch_size) via replace

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `dataclasses-replace-args`. Running the final beacon cell reports progress against the `Config: dataclasses.replace args` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Config: dataclasses.replace args` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dataclasses-replace-args`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dataclasses-replace-args"
DD_SUBTOPIC = "Config: dataclasses.replace args"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `dataclasses.replace` — 2-D grid sweep

Ex1 swept ONE axis (LR). The natural deepening is the 2-D grid: for every `(lr, batch_size)` cross-pair, build one variant. The idiom is a nested comprehension that passes BOTH overrides into a single `replace` call:

```python
variants = [
    replace(base, lr=lr, batch_size=bs)
    for lr in lrs
    for bs in bss
]
```

**Why ONE `replace` per pair (not nested replaces).** `replace(replace(base, lr=lr), batch_size=bs)` works but pays two `__post_init__` runs per variant and produces an intermediate object. The single-call form is the same result, one validation, one allocation.

**Ordering convention.** Outer loop is the SLOWER-varying axis (matches numpy / itertools.product) — first vary `bs` for each fixed `lr`, then advance `lr`. Match this when feeding a sweep harness so logs are sorted in a predictable order.

### Exercise 2 — 2-D grid sweep over (lr, batch_size) via replace

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply a single `dataclasses.replace` call inside a nested comprehension to produce a 2-D Cartesian-product sweep of training-args variants ordered row-major over (lr, batch_size).
> Keywords: dataclass, replace, grid-sweep, cartesian-product
> ```

**KCs targeted:** `dataclasses-replace-keyword-overrides`, `grid-sweep-row-major-order`

Implement `ex2_make_grid_sweep(base, lrs, batch_sizes)`.

For every `(lr, bs)` pair in the Cartesian product, build a fresh `TrainingArgs` with BOTH overrides set in one `replace` call. Order is **row-major**: outer loop is `lr`, inner loop is `batch_size`. So `lrs=[a, b], batch_sizes=[1, 2, 3]` produces 6 variants in order: `(a,1), (a,2), (a,3), (b,1), (b,2), (b,3)`.

Constraints:
- Total variants = `len(lrs) * len(batch_sizes)`.
- Each variant uses ONE `replace` call (not nested replaces).
- `base` must NOT be mutated.
- `__post_init__` validation must still fire (a bad lr   anywhere in the grid raises `ValueError`).
- Empty axis -> empty grid.

In [ ]:
from dataclasses import dataclass, replace

@dataclass
class TrainingArgs:
    lr: float = 1e-3
    batch_size: int = 32
    epochs: int = 10
    optimizer_name: str = 'adam'

    def __post_init__(self):
        if self.lr <= 0:
            raise ValueError(f'lr must be > 0, got {self.lr}')
        if self.batch_size < 1:
            raise ValueError(f'batch_size must be >= 1, got {self.batch_size}')

def ex2_make_grid_sweep(base, lrs, batch_sizes):
    return [
        replace(base, lr=lr, batch_size=bs)
        for lr in lrs
        for bs in batch_sizes
    ]


<details><summary>Solution</summary>

```python
from dataclasses import dataclass, replace

@dataclass
class TrainingArgs:
    lr: float = 1e-3
    batch_size: int = 32
    epochs: int = 10
    optimizer_name: str = 'adam'

    def __post_init__(self):
        if self.lr <= 0:
            raise ValueError(f'lr must be > 0, got {self.lr}')
        if self.batch_size < 1:
            raise ValueError(f'batch_size must be >= 1, got {self.batch_size}')

def ex2_make_grid_sweep(base, lrs, batch_sizes):
    return [
        replace(base, lr=lr, batch_size=bs)
        for lr in lrs
        for bs in batch_sizes
    ]
```

**Why row-major over column-major.** Matches `itertools.product(lrs, batch_sizes)` and numpy `meshgrid(..., indexing='ij').reshape(-1, 2)`. When you log sweep results, the natural sort key is `(lr, bs)` tuples in lexicographic order — that's exactly what row-major produces.

**One `replace` per pair, two kwargs.** `replace(base, lr=lr, batch_size=bs)` passes BOTH overrides in one call — one allocation, one `__post_init__` invocation. Nested calls `replace(replace(base, lr=lr), batch_size=bs)` work but double the cost.

**Empty axis -> empty grid.** Either factor zero makes the Cartesian product empty — the comprehension naturally returns `[]` because the outer or inner loop has nothing to iterate. No special case needed.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()